# AI for Engineers Lab 3: Constraint Satisfaction and Backtracking

## The Exam Scheduling Challenge

The engineering school must place four exams into three time slots. Some exams share students, so they cannot run at the same time. We will use this one scenario to learn **variables**, **domains**, **constraints**, and **backtracking**.

**Estimated time:** 30 minutes  
**Requirements:** Python 3 only. No extra packages.

By the end of the lab, you should be able to:

1. model a small problem as a constraint satisfaction problem (CSP),
2. test whether a partial assignment satisfies the constraints,
3. solve the CSP using backtracking, and
4. explain when and why the algorithm backtracks.

> Run each code cell with **Shift+Enter**. Complete the questions and TODOs as you go.

**Andrew ID:** cuwatwem

### The main idea

The goal is to give every exam a time slot without breaking any rules. An exam
is a **variable**, its possible slots are its **domain**, and the scheduling
rules are **constraints**. Backtracking tries a choice and undoes it if that
choice cannot lead to a full timetable.

---
## Part 1: Model the timetable as a CSP (about 6 minutes)

A CSP has three ingredients:

- **Variables:** the decisions we must make.
- **Domains:** the possible values for each variable.
- **Constraints:** rules that every solution must satisfy.

In this lab, each exam is a variable and its value is a time slot.

In [10]:
# The four exam variables
exams = ["Calculus", "Circuits", "Programming", "Mechanics"]

# Every exam starts with the same possible time slots.
slots = ["Mon 09:00", "Mon 14:00", "Tue 09:00"]

# Each exam gets its own list, so changing one domain does not change the others.
domains = {exam: slots.copy() for exam in exams}

# These pairs share students, so they need different time slots.
conflicts = {
    "Calculus": {"Circuits", "Programming"},
    "Circuits": {"Calculus", "Mechanics"},
    "Programming": {"Calculus", "Mechanics"},
    "Mechanics": {"Circuits", "Programming"},
}

print("Variables:", exams)
print("Domain of Calculus:", domains["Calculus"])
print("Conflicts:")
for exam in exams:
    print(f"  {exam}: {sorted(conflicts[exam])}")


Variables: ['Calculus', 'Circuits', 'Programming', 'Mechanics']
Domain of Calculus: ['Mon 09:00', 'Mon 14:00', 'Tue 09:00']
Conflicts:
  Calculus: ['Circuits', 'Programming']
  Circuits: ['Calculus', 'Mechanics']
  Programming: ['Calculus', 'Mechanics']
  Mechanics: ['Circuits', 'Programming']


### Question 1

In this timetable, what are the variables, domains, and constraints? Why is the
rule “Calculus cannot be at the same time as Circuits” a **binary** constraint?

**Answer:**

- The variables are **Calculus, Circuits, Programming, and Mechanics**.
- Each exam starts with the domain **Mon 09:00, Mon 14:00, and Tue 09:00**.
- The constraints say that exams with shared students must have different slots.
  The four pairs are Calculus–Circuits, Calculus–Programming,
  Circuits–Mechanics, and Programming–Mechanics.

The Calculus–Circuits rule is binary because it connects **two variables**.
Binary does not mean that an exam has only two possible slots. Exams without
a conflict, like Circuits and Programming, can use the same slot.


---
## Part 2: Check a partial assignment (about 7 minutes)

Backtracking builds a solution one exam at a time. Before placing an exam, it checks whether the proposed slot conflicts with exams already placed.

The function below returns `True` when a proposed choice is allowed.

### TODO 2

Inside `is_consistent`, write the condition that detects whether a conflicting exam already has the proposed slot. Replace the `raise NotImplementedError` line with the hinted `if` statement and its body.

In [2]:
def is_consistent(exam, slot, assignment, trace=False):
    """Check the proposed slot against exams that are already placed."""
    for other_exam in sorted(conflicts[exam]):
        if assignment.get(other_exam) == slot:
            if trace:
                print(f"Reject {exam} at {slot}: {other_exam} is already there.")
            return False
    return True


# Only Calculus has been placed so far.
partial = {"Calculus": "Mon 09:00"}

for candidate in slots:
    allowed = is_consistent("Circuits", candidate, partial, trace=True)
    print(f"Circuits at {candidate}: {allowed}")


Reject Circuits at Mon 09:00: Calculus is already there.
Circuits at Mon 09:00: False
Circuits at Mon 14:00: True
Circuits at Tue 09:00: True


### Question 2

Why can Circuits use two of the slots but not `Mon 09:00`? Would the result
change if the partial assignment were empty?

**Answer:**

Calculus already uses `Mon 09:00`, and Calculus and Circuits share students.
Putting them together would break their constraint. The other two slots do
not clash with any assigned exam, so both are allowed.

If the partial assignment were empty, all three slots would pass this check.
There would be no assigned exam to clash with yet. 


---
## Part 3: Solve with backtracking (about 10 minutes)

Backtracking uses depth-first search over assignments:

1. choose an unassigned exam,
2. try one value from its domain,
3. continue if the choice is consistent,
4. undo the choice when it leads to a dead end.

### TODO 3

Complete the four short lines marked `TODO 3a` through `TODO 3d`: select an exam, record a choice, make the recursive call, and undo a failed choice. The surrounding search logic is already provided.

In [3]:
def backtracking_search(assignment=None, depth=0, trace=True, stats=None):
    """Return (solution, stats), or None if no full timetable is possible."""
    if assignment is None:
        assignment = {}
    if stats is None:
        stats = {"attempts": 0, "backtracks": 0}

    # Every exam has a slot, so the timetable is complete.
    if len(assignment) == len(exams):
        return assignment.copy(), stats

    # Choose the first exam that still needs a slot.
    exam = next(e for e in exams if e not in assignment)

    for slot in domains[exam]:
        stats["attempts"] += 1
        if trace:
            print("  " * depth + f"Try {exam} = {slot}")

        if is_consistent(exam, slot, assignment):
            assignment[exam] = slot

            # Try to place the remaining exams using this choice.
            result = backtracking_search(assignment, depth + 1, trace, stats)
            if result is not None:
                return result

            # This choice did not work, so remove it and try the next slot.
            del assignment[exam]

    stats["backtracks"] += 1
    if trace:
        print("  " * depth + f"Backtrack from {exam}")
    return None


result = backtracking_search()
solution, stats = result

print()
print("Solved timetable:")
for exam in exams:
    print(f"  {exam:12s} -> {solution[exam]}")
print("Search statistics:", stats)


Try Calculus = Mon 09:00
  Try Circuits = Mon 09:00
  Try Circuits = Mon 14:00
    Try Programming = Mon 09:00
    Try Programming = Mon 14:00
      Try Mechanics = Mon 09:00

Solved timetable:
  Calculus     -> Mon 09:00
  Circuits     -> Mon 14:00
  Programming  -> Mon 14:00
  Mechanics    -> Mon 09:00
Search statistics: {'attempts': 6, 'backtracks': 0}


In [4]:
def validate_solution(solution):
    """Check completeness, legal domain values, and all conflict constraints."""
    if solution is None or set(solution) != set(exams):
        return False
    if any(solution[e] not in domains[e] for e in exams):
        return False
    for exam in exams:
        for other_exam in conflicts[exam]:
            if solution[exam] == solution[other_exam]:
                return False
    return True


print("Valid solution:", validate_solution(solution))

Valid solution: True


### Question 3

What line of code **undoes** a decision? In your own words, explain why that
line is necessary.

**Answer:**

The line is `del assignment[exam]`. It removes the slot chosen for that exam
when the recursive call cannot finish the timetable. A recursive call means
the function calls itself to solve the remaining part of the problem.

Removing the failed choice lets the solver try another slot with a clean
partial timetable. If the old choice stayed in the dictionary, later calls
could treat it as a decision that was still valid.

The first run does not need this line because its first accepted choices lead
to a solution. The optional extension below shows a case where it is needed.


---
## Part 4: Change a domain and solve again (about 5 minutes)

The lecturer for Mechanics is unavailable on Monday morning. This is a **unary constraint** because it restricts only one variable.

### TODO 4

Write one line that removes `Mon 09:00` from the domain of Mechanics. Then rerun the solver and compare the new timetable with the first result.

In [5]:
# Mechanics cannot use Monday morning.
# Start from slots so running this cell again gives the same domain.
domains["Mechanics"] = [slot for slot in slots if slot != "Mon 09:00"]

new_result = backtracking_search(trace=False)

if new_result is None:
    print("No timetable satisfies all constraints.")
else:
    new_solution, new_stats = new_result
    print("Updated timetable:")
    for exam in exams:
        print(f"  {exam:12s} -> {new_solution[exam]}")
    print("Valid solution:", validate_solution(new_solution))
    print("Search statistics:", new_stats)


Updated timetable:
  Calculus     -> Mon 09:00
  Circuits     -> Mon 14:00
  Programming  -> Mon 14:00
  Mechanics    -> Tue 09:00
Valid solution: True
Search statistics: {'attempts': 7, 'backtracks': 0}


### Question 4

How did reducing Mechanics' domain affect the search? Give one example of a
real scheduling rule that could be represented as a unary constraint and a
binary constraint.

**Answer:**

The new domain removes Monday morning from Mechanics. The table below compares
the two runs.

| Exam | Original timetable | Mechanics cannot use Monday morning |
|---|---|---|
| Calculus | Mon 09:00 | Mon 09:00 |
| Circuits | Mon 14:00 | Mon 14:00 |
| Programming | Mon 14:00 | Mon 14:00 |
| Mechanics | Mon 09:00 | Tue 09:00 |

The original run made **6 attempts** and the new run made
**7 attempts**. Both had **0 backtracks**. In the new run, the solver
first tries `Mon 14:00` for Mechanics, but that clashes with Circuits and
Programming. It then uses `Tue 09:00`.

The smaller domain removes the original working choice. For this search order,
that means one more attempt, not fewer. Neither run needs to undo an accepted
decision.

- **Unary example:** Mechanics must be on Tuesday because its lecturer is
  unavailable on Monday. This limits one exam's domain.
- **Binary example:** Calculus and Circuits cannot run together because some
  students take both. This connects two exams.


---
# Wrap-up

You used one exam-scheduling problem to connect the main CSP ideas:

- **Variables:** exams that need decisions.
- **Domains:** time slots each exam may use.
- **Constraints:** allowed combinations of assignments.
- **Backtracking:** try a value, continue when consistent, and undo a failed choice.

